
# 02 — The Real Results: Harmonic Oscillator, and How to Statistically Trust Any of This

**Prerequisite:** Notebooks `00` and `01`.

Notebook `01` showed you a *tiny, fast* version of Method C so you could watch
the mechanism work. This notebook does two things:

1. Shows you the **real, full-scale, 5-seed results** the project actually
   reports (loaded straight from the repo's result files — nothing
   re-computed or approximated).
2. Teaches you **how to tell whether a result is real or just noise**, using
   the Friedman and Nemenyi tests — the two statistical tools used
   everywhere else in this project.


In [ ]:

import sys, os, json
import numpy as np

REPO_PATH = None
_candidates = [REPO_PATH, "HPO-HMC", os.path.join("..", "HPO-HMC"), "."]
for _c in _candidates:
    if _c and os.path.isdir(os.path.join(_c, "results")):
        REPO_PATH = _c
        break
if REPO_PATH is None:
    raise FileNotFoundError("Couldn't find the HPO-HMC repo. Set REPO_PATH manually.")
print(f"Using repo at: {os.path.abspath(REPO_PATH)}")

def load_json(*parts):
    with open(os.path.join(REPO_PATH, *parts)) as f:
        return json.load(f)



## 1. The harmonic oscillator: real 5-seed numbers

Recall the task: reconstruct a known energy landscape from data, where the
*true* answer is known exactly (unlike real-world data), so error can be
measured with no ambiguity.


In [ ]:

method_c_seeds = load_json("results", "method_c_fixed_results.json")

mse_vals = [r["best_val_loss"] for r in method_c_seeds]
r2_vals  = [r["r2"] for r in method_c_seeds]
time_vals = [r["train_time"] for r in method_c_seeds]

print("Method C (Unified HHD-ABBO) -- 5 real seeds, harmonic oscillator benchmark")
print(f"  Best Val. MSE : {np.mean(mse_vals):.5f} +/- {np.std(mse_vals):.5f}")
print(f"  R^2           : {np.mean(r2_vals):.5f} +/- {np.std(r2_vals):.5f}")
print(f"  Wall time (s) : {np.mean(time_vals):.1f} +/- {np.std(time_vals):.1f}")
print()
print("Compare against the paper's reported table (verified, same numbers):")
print("  Method A (HHD):    MSE = 0.2439 +/- 0.1627,  R^2 = 0.9785 +/- 0.0158,  time = 26.6s")
print("  Method B (ABBO):   MSE = 0.0952 +/- 0.0051,  R^2 = 0.9984 +/- 0.0005,  time = 99.9s")
print("  Method C (Unified):MSE = 0.0033 +/- 0.0001,  R^2 = 0.99994 +/- 0.00001, time = 85.6s")
print()
print("Method C is ~74x lower MSE than Method A, at ~3.2x the wall-clock cost.")



## 2. How do you know a result like "Optuna TPE has rank 1.36" is real?

The project also tested five methods (Random Search, Optuna TPE, and Methods
A/B/C) across **11 different real HPO benchmarks** (HPOBench, HPOLib,
NAS-Bench-201). Some of those benchmarks have accuracies around 45%, others
around 95% — a simple average across datasets would let one weird dataset
dominate.

**The fix: rank the 5 methods within each dataset (1st through 5th place),
then average the ranks.** This is fair regardless of each dataset's scale.
Let's build this from scratch on a toy example first, so the mechanism is
completely transparent, then apply it to the real data.


In [ ]:

from scipy import stats

# --- Toy example: 4 fake "datasets", 3 fake "methods" ---
# Each row = one dataset; each column = one method's score (higher = better).
np.random.seed(42)
toy_scores = np.array([
    [0.91, 0.95, 0.60],   # dataset 1
    [0.70, 0.85, 0.55],   # dataset 2
    [0.88, 0.92, 0.58],   # dataset 3
    [0.65, 0.80, 0.50],   # dataset 4
])
method_names = ["Method X", "Method Y (best-looking)", "Method Z"]

# Rank each row (1 = best). Higher score = better, so we rank the NEGATIVE score.
ranks = np.array([stats.rankdata(-row) for row in toy_scores])
print("Per-dataset ranks (1 = best):")
for i, row in enumerate(ranks):
    print(f"  dataset {i+1}: {dict(zip(method_names, row))}")

avg_ranks = ranks.mean(axis=0)
print("\nAverage rank per method:")
for name, r in zip(method_names, avg_ranks):
    print(f"  {name}: {r:.2f}")



### Is "Method Y" really better, or could this be luck?

This is exactly what the **Friedman test** answers. Its logic: *if all three
methods were secretly equally good*, the ranks in each row would be randomly
shuffled, and the average ranks across many datasets should come out close to
equal. The Friedman test measures how far the observed average ranks are from
that "all equal" scenario, and gives you a p-value: the probability of seeing
a spread this large purely by chance if there were truly no difference.


In [ ]:

stat, p = stats.friedmanchisquare(*[toy_scores[:, j] for j in range(toy_scores.shape[1])])
print(f"Friedman chi^2 = {stat:.3f}, p = {p:.4f}")
if p < 0.05:
    print("p < 0.05 -> reject 'all methods are equal'. There IS a real difference somewhere.")
else:
    print("p >= 0.05 -> can't rule out that this is just noise, with only 4 toy datasets.")



The Friedman test tells you *whether* a difference exists somewhere among the
methods — but not *which pairs* are actually different. For that, you need
the **Nemenyi post-hoc test**, which computes a **Critical Difference (CD)**:
any two methods whose average ranks differ by more than the CD are
confidently different; if the gap is smaller, you can't be sure.

Now let's apply this exact machinery to the **real** 11-dataset result from
the project.


In [ ]:

# The real, verified per-dataset ranks from the paper (Table 14, HO_main.tex) --
# 11 datasets x 5 methods (Random, Optuna TPE, Method A, Method B, Method C).
real_ranks = np.array([
    # Rand, TPE, A, B, C
    [2, 3, 5, 1, 4],   # Australian
    [3, 1, 4, 5, 2],   # Blood Transfusion
    [4, 2, 3, 5, 1],   # Vehicle
    [3, 1, 5, 4, 2],   # Segment
    [5, 1, 4, 2, 3],   # Naval Propulsion
    [3, 1, 2, 4, 5],   # Parkinsons
    [5, 2, 3, 1, 4],   # Protein Structure
    [5, 1, 4, 3, 2],   # Slice Localization
    [3, 1, 5, 2, 4],   # CIFAR-10 (NAS-Bench-201)
    [3, 1, 5, 2, 4],   # CIFAR-100 (NAS-Bench-201)
    [4, 1, 5, 2, 3],   # ImageNet-16 (NAS-Bench-201)
])
real_method_names = ["Random Search", "Optuna TPE", "Method A (HHD)", "Method B (ABBO)", "Method C (Unified)"]

avg_ranks_real = real_ranks.mean(axis=0)
print("Average rank across all 11 real datasets:")
for name, r in sorted(zip(real_method_names, avg_ranks_real), key=lambda x: x[1]):
    print(f"  {name:20s}: {r:.2f}")

stat, p = stats.friedmanchisquare(*[real_ranks[:, j] for j in range(5)])
print(f"\nFriedman test: chi^2 = {stat:.3f}, p = {p:.2e}")
print("This matches the paper's reported p = 7.92e-4 -- highly significant:")
print("there IS a genuine difference among these 5 methods.")


In [ ]:

import scipy.stats as ss

N, k = real_ranks.shape  # 11 datasets, 5 methods
# Nemenyi Critical Difference formula:
#   CD = q_alpha * sqrt(k*(k+1) / (6*N))
# q_alpha for k=5 methods at alpha=0.05 is a tabulated constant (~2.728)
q_alpha = 2.728
CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))
print(f"Nemenyi Critical Difference (alpha=0.05, k={k} methods, N={N} datasets): CD = {CD:.3f}")
print("This matches the paper's reported CD = 1.84.\n")

best_name, best_rank = min(zip(real_method_names, avg_ranks_real), key=lambda x: x[1])
print(f"Best average rank: {best_name} ({best_rank:.2f})\n")
print("Gap from the best method, and whether it's bigger than CD:")
for name, r in zip(real_method_names, avg_ranks_real):
    if name == best_name:
        continue
    gap = r - best_rank
    verdict = "SIGNIFICANTLY WORSE" if gap > CD else "not statistically distinguishable"
    print(f"  {name:20s} gap = {gap:.2f}  ({verdict})")



### The honest conclusion this produces

Optuna TPE has the best average rank (1.36). But statistically:
- **Method B and Method C are NOT proven worse than Optuna** — their rank
  gaps (1.46 and 1.73) are both smaller than the Critical Difference (1.84).
- **Random Search and Method A ARE proven worse than Optuna** — their gaps
  (2.28 and 2.73) exceed the CD.

This is a genuinely important distinction: "Optuna wins on average" and
"Optuna is proven better than Method C" are *different claims*. Only the
first is fully supported here — and it would have been easy (and wrong) to
oversell the second. This is exactly the kind of restraint this project's
paper exercises throughout.

**Next notebook (`03`):** applying all of this — Method C, real baselines,
and honest significance testing — to genuine clinical diagnostic data, and
digging into a real surprise the project found there.
